<a href="https://colab.research.google.com/github/cebrailbagatarhan/yapay-zeka-sistemi/blob/main/modern_llm_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Modern LLM - Sıfırdan Eğitim Notebook'u

**Sıfırdan yazılmış modern bir LLM'i Google Colab üzerinde eğitin.**

## Mimari Özellikler
- **RMSNorm** (Pre-normalization)
- **Rotary Position Embeddings (RoPE)** θ=500,000
- **Grouped Query Attention (GQA)** — 4:1 ratio
- **SwiGLU Activation** — Gate + Up + Down projections
- **Flash Attention** — PyTorch 2.0+ SDPA
- **KV-Cache** — Verimli autoregressive inference
- **Chain-of-Thought (CoT)** — `<think>...</think>` reasoning

## Model Boyutları
| Model | Params | GPU | Hidden | Layers |
|-------|--------|-----|--------|--------|
| Nano  | ~30M   | T4  | 512    | 8      |
| Small | ~100M  | L4  | 768    | 12     |
| Medium| ~350M  | A100| 1024   | 24     |
| Large | ~1.3B  | H100| 2048   | 24     |

## Adımlar
1. **Kurulum** — GPU kontrolü + bağımlılıklar
2. **Veri Hazırlama** — Turkish + English datasets
3. **Tokenizer Eğitimi** — BPE tokenizer
4. **Model Oluşturma** — GPU'ya uygun config
5. **Pre-training** — Causal LM
6. **SFT (Instruction Tuning)** — Chat format eğitimi
7. **CoT Fine-tuning** — Reasoning eğitimi
8. **Değerlendirme** — Örnek çıktılar + metrikler
9. **Kaydetme** — Model + tokenizer export

---
**Yazar:** Cebrail Bağatar Han | **Versiyon:** 1.0

## 1️⃣ GPU Kontrolü ve Kurulum

In [ ]:
# ═══════════════════════════════════════════════════════
# GPU Kontrolü ve Ortam Tespiti
# ═══════════════════════════════════════════════════════
import torch
import os
import sys
import gc

print("🔍 GPU KONTROLÜ")
print("=" * 60)

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
print(f"📍 Ortam: {'Google Colab' if IN_COLAB else 'Lokal'}")
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")

GPU_TYPE = "CPU"
GPU_MEMORY_GB = 0

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_mem / 1024**3
    
    if 'H100' in gpu_name:
        GPU_TYPE = 'H100'
    elif 'A100' in gpu_name:
        GPU_TYPE = 'A100'
    elif 'L4' in gpu_name:
        GPU_TYPE = 'L4'
    elif 'T4' in gpu_name:
        GPU_TYPE = 'T4'
    elif 'V100' in gpu_name:
        GPU_TYPE = 'V100'
    else:
        GPU_TYPE = 'OTHER'
    
    print(f"\n🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {GPU_MEMORY_GB:.1f} GB")
    print(f"📊 CUDA: {torch.version.cuda}")
    print(f"⚡ BF16 Desteği: {'✅' if torch.cuda.is_bf16_supported() else '❌'}")
    print(f"🏷️ GPU Tipi: {GPU_TYPE}")
else:
    print("\n⚠️ GPU bulunamadı! CPU modunda çalışacak (çok yavaş).")

print(f"\n{'=' * 60}")
print(f"{'✅ GPU hazır!' if GPU_TYPE != 'CPU' else '⚠️ GPU yok, küçük model deneyin.'}")

In [ ]:
# ═══════════════════════════════════════════════════════
# Bağımlılıkları Kur
# ═══════════════════════════════════════════════════════
print("📦 Gerekli kütüphaneler kuruluyor...")

!pip install -q sentencepiece datasets wandb accelerate

# Colab'da repo'yu klonla
if IN_COLAB:
    if not os.path.exists('yapay-zeka-sistemi'):
        !git clone https://github.com/cebrailbagatarhan/yapay-zeka-sistemi.git
    os.chdir('yapay-zeka-sistemi')
    print(f"📂 Çalışma dizini: {os.getcwd()}")
else:
    print(f"📂 Çalışma dizini: {os.getcwd()}")

# Modern LLM modülünü import et
sys.path.insert(0, os.getcwd())

print("✅ Kurulum tamamlandı!")

In [ ]:
# ═══════════════════════════════════════════════════════
# Tüm Modülleri İçe Aktar
# ═══════════════════════════════════════════════════════
from modern_llm.config import ModelConfig, TrainingConfig, PRESET_CONFIGS, GPU_PRESETS, get_config_for_gpu
from modern_llm.model.transformer import ModernLLMForCausalLM
from modern_llm.tokenizer import ModernTokenizer
from modern_llm.training.trainer import Trainer
from modern_llm.training.dataset import (
    TextDataset, ChatDataset, CoTDataset,
    load_dataset_from_json, load_huggingface_dataset,
    convert_to_chat_format, convert_cot_format, create_data_collator
)
from modern_llm.inference.generator import TextGenerator, ChatInterface
from modern_llm.cot.engine import ChainOfThoughtEngine
from modern_llm.utils import (
    get_device, get_gpu_info, detect_optimal_config,
    count_parameters, format_params, estimate_memory,
    set_seed, load_json, save_json,
    print_model_summary
)

print("✅ Tüm modüller başarıyla yüklendi!")

## 2️⃣ Konfigürasyon ve GPU Optimizasyonu

In [ ]:
# ═══════════════════════════════════════════════════════
# GPU'ya Göre Otomatik Konfigürasyon
# ═══════════════════════════════════════════════════════
SEED = 42
set_seed(SEED)
device = get_device()

# GPU'ya göre optimal konfigürasyon seç
if GPU_TYPE in GPU_PRESETS:
    model_config, train_config = get_config_for_gpu(GPU_TYPE)
    print(f"✅ {GPU_TYPE} GPU için optimize edilmiş konfigürasyon seçildi")
else:
    # Varsayılan: Nano model
    model_config = PRESET_CONFIGS["nano"]
    train_config = TrainingConfig(
        batch_size=2,
        max_seq_length=512,
        mixed_precision="fp16" if torch.cuda.is_available() else "no",
        gradient_accumulation_steps=8,
    )
    print(f"⚠️ Bilinmeyen GPU, Nano model kullanılacak")

# Eğitim ayarlarını güncelle
train_config.num_epochs = 3
train_config.learning_rate = 3e-4
train_config.warmup_ratio = 0.1
train_config.weight_decay = 0.1
train_config.max_grad_norm = 1.0
train_config.logging_steps = 10
train_config.eval_steps = 100
train_config.save_steps = 200
train_config.output_dir = "checkpoints"
train_config.use_wandb = False  # True yaparak wandb kullanabilirsiniz

print(f"\n{'=' * 60}")
print(f"📐 MODEL KONFİGÜRASYONU")
print(f"{'=' * 60}")
print(f"  Model: {model_config.model_name}")
print(f"  Hidden Size: {model_config.hidden_size}")
print(f"  Layers: {model_config.num_layers}")
print(f"  Attention Heads: {model_config.num_attention_heads}")
print(f"  KV Heads: {model_config.num_kv_heads} (GQA ratio: {model_config.num_attention_heads // model_config.num_kv_heads}:1)")
print(f"  FFN Size: {model_config.intermediate_size}")
print(f"  Max Seq Length: {model_config.max_position_embeddings}")
print(f"  Vocab Size: {model_config.vocab_size}")
print(f"  RoPE θ: {model_config.rope_theta}")

print(f"\n{'=' * 60}")
print(f"🏋️ EĞİTİM KONFİGÜRASYONU")
print(f"{'=' * 60}")
print(f"  Batch Size: {train_config.batch_size}")
print(f"  Gradient Accumulation: {train_config.gradient_accumulation_steps}")
print(f"  Effective Batch: {train_config.batch_size * train_config.gradient_accumulation_steps}")
print(f"  Learning Rate: {train_config.learning_rate}")
print(f"  Mixed Precision: {train_config.mixed_precision}")
print(f"  Epochs: {train_config.num_epochs}")
print(f"  Max Seq Length: {train_config.max_seq_length}")

## 3️⃣ Veri Hazırlama

Eğitim verisini üç aşamada hazırlıyoruz:
1. **Pre-training corpus** — Büyük metin koleksiyonu (next-token prediction)
2. **SFT (Instruction) verisi** — Chat formatında instruction-following
3. **CoT verisi** — Chain-of-thought reasoning örnekleri

In [ ]:
# ═══════════════════════════════════════════════════════
# 3a. Pre-training Verisi
# ═══════════════════════════════════════════════════════
print("📚 Pre-training verisi hazırlanıyor...")

pretrain_texts = []

# --- 1. HuggingFace'den Türkçe corpus yükle ---
print("\n📥 HuggingFace'den Türkçe veri çekiliyor...")

# Türkçe Wikipedia benzeri veri
try:
    hf_data = load_huggingface_dataset(
        "alibayram/turkish_instructions_150k",
        split="train",
        max_samples=5000,
    )
    for item in hf_data:
        if "output" in item and item["output"]:
            pretrain_texts.append(item["output"])
        if "instruction" in item and item["instruction"]:
            pretrain_texts.append(item["instruction"])
    print(f"  ✅ alibayram/turkish_instructions_150k: {len(hf_data)} örnek")
except Exception as e:
    print(f"  ⚠️ HF dataset yüklenemedi: {e}")

# --- 2. Lokal veri dosyaları ---
local_data_paths = [
    "data/training/conversational_dataset.json",
    "data/training/reasoning_chat_dataset.json",
    "data/training/code_examples_dataset.json",
    "data/training/turkish_general_dataset.json",
]

for path in local_data_paths:
    if os.path.exists(path):
        try:
            data = load_json(path)
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict):
                        for key in ["output", "response", "content", "text", "answer"]:
                            if key in item and item[key]:
                                pretrain_texts.append(str(item[key]))
                                break
                    elif isinstance(item, str):
                        pretrain_texts.append(item)
            print(f"  ✅ {path}: yüklendi")
        except Exception as e:
            print(f"  ⚠️ {path}: {e}")

# --- 3. Minimal veri garanti ---
if len(pretrain_texts) < 100:
    print("\n📝 Minimum veri sağlanıyor (dahili örnekler)...")
    turkish_texts = [
        "Türkiye, Avrupa ve Asya kıtalarının kesişim noktasında yer alan bir ülkedir. Başkenti Ankara olan Türkiye, zengin tarihsel ve kültürel mirasa sahiptir.",
        "Yapay zeka, makinelerin insan benzeri düşünme ve öğrenme yeteneğine sahip olmasını sağlayan bir bilim dalıdır. Derin öğrenme, doğal dil işleme ve bilgisayarlı görü bu alanın önemli alt dallarıdır.",
        "Python programlama dili, basit sözdizimi ve geniş kütüphane desteği ile dünyada en çok kullanılan programlama dillerinden biridir. Veri bilimi, web geliştirme ve yapay zeka alanlarında yaygın olarak kullanılır.",
        "İstanbul, Türkiye'nin en kalabalık şehri olup Boğaziçi ile Avrupa ve Asya'yı birbirine bağlar. Tarih boyunca Roma, Bizans ve Osmanlı İmparatorluklarına başkentlik yapmıştır.",
        "Transformer mimarisi, 2017 yılında 'Attention is All You Need' makalesi ile tanıtılmıştır. Self-attention mekanizması sayesinde paralel işleme ve uzun menzilli bağımlılıkları yakalama konusunda devrim yaratmıştır.",
        "Makine öğrenmesi, verilerdeki örüntüleri otomatik olarak keşfeden algoritmaların geliştirilmesiyle ilgilenir. Gözetimli öğrenme, gözetimsiz öğrenme ve pekiştirmeli öğrenme üç temel yaklaşımdır.",
        "Doğal dil işleme, bilgisayarların insan dilini anlaması ve üretmesi için geliştirilen teknolojiler bütünüdür. Metin sınıflandırma, duygu analizi, makine çevirisi ve soru cevaplama başlıca uygulama alanlarıdır.",
        "Nöral ağlar, insan beynindeki sinir hücrelerinden esinlenerek tasarlanmış matematiksel modellerdir. Giriş katmanı, gizli katmanlar ve çıkış katmanından oluşur.",
        "Büyük dil modelleri (LLM), milyarlarca parametre ile eğitilen devasa yapay sinir ağlarıdır. GPT, Claude, Gemini ve LLaMA gibi modeller bu kategoriye girer.",
        "Gradient descent, kayıp fonksiyonunu minimize etmek için kullanılan optimizasyon algoritmasıdır. Learning rate, momentum ve adaptif öğrenme oranları gibi varyantları bulunur.",
        "Attention mekanizması, bir dizideki her elemanın diğer elemanlarla ilişkisini hesaplayan bir tekniktir. Query, Key ve Value matrislerinin çarpımından oluşur.",
        "Tokenization, metni küçük parçalara (tokenlara) ayırma işlemidir. BPE (Byte-Pair Encoding), WordPiece ve SentencePiece yaygın kullanılan tokenization yöntemleridir.",
        "Transfer öğrenme, bir görev için eğitilmiş modelin başka bir görevde kullanılmasıdır. Fine-tuning ve feature extraction iki temel transfer öğrenme yaklaşımıdır.",
        "Veri ön işleme, ham verilerin makine öğrenmesi modellerine uygun hale getirilmesi sürecidir. Eksik veri doldurma, normalizasyon ve özellik mühendisliği adımlarını içerir.",
        "Backpropagation, yapay sinir ağlarında ağırlıkların güncellenmesi için kullanılan algoritmadır. Zincir kuralı kullanarak her katmandaki gradyanları hesaplar.",
        "Convolutional Neural Networks (CNN), özellikle görüntü tanıma ve işleme için tasarlanmış derin öğrenme mimarisidir. Konvolüsyon, pooling ve fully connected katmanlardan oluşur.",
        "Recurrent Neural Networks (RNN), sıralı verileri işlemek için geliştirilmiş sinir ağı mimarisidir. LSTM ve GRU gibi varyantları, uzun vadeli bağımlılık sorununu çözer.",
        "Regularization teknikleri, modelin aşırı öğrenmesini (overfitting) önlemek için kullanılır. Dropout, L1/L2 regularization ve early stopping yaygın yöntemlerdir.",
        "Batch normalization, her mini-batch için aktivasyonları normalize ederek eğitimi stabilize eden bir tekniktir. Eğitim hızını artırır ve daha yüksek öğrenme oranları kullanılmasına izin verir.",
        "Generative AI, yeni içerik üreten yapay zeka sistemleridir. Metin, görüntü, ses ve video üretme yeteneğine sahip modeller bu kategoride yer alır.",
    ]
    pretrain_texts.extend(turkish_texts)

# Kısa metinleri filtrele
pretrain_texts = [t for t in pretrain_texts if len(t.strip()) > 50]

print(f"\n{'=' * 60}")
print(f"📊 PRE-TRAINING VERİ ÖZETİ")
print(f"{'=' * 60}")
print(f"  Toplam metin sayısı: {len(pretrain_texts)}")
print(f"  Toplam karakter: {sum(len(t) for t in pretrain_texts):,}")
print(f"  Ortalama metin uzunluğu: {sum(len(t) for t in pretrain_texts) / max(len(pretrain_texts), 1):.0f} karakter")

In [ ]:
# ═══════════════════════════════════════════════════════
# 3b. SFT (Instruction-Following) Verisi
# ═══════════════════════════════════════════════════════
print("📚 SFT (Instruction) verisi hazırlanıyor...")

sft_data = []

# --- 1. HuggingFace'den instruction verisi ---
print("\n📥 HuggingFace instruction verisi çekiliyor...")

try:
    hf_instructions = load_huggingface_dataset(
        "alibayram/turkish_instructions_150k",
        split="train",
        max_samples=3000,
    )
    sft_from_hf = convert_to_chat_format(
        hf_instructions,
        instruction_key="instruction",
        output_key="output",
        input_key="input",
        system_prompt="Sen yardımcı bir yapay zeka asistanısın. Soruları detaylı ve doğru bir şekilde yanıtla.",
    )
    sft_data.extend(sft_from_hf)
    print(f"  ✅ alibayram/turkish_instructions_150k → {len(sft_from_hf)} sohbet")
except Exception as e:
    print(f"  ⚠️ HF instruction verisi yüklenemedi: {e}")

# --- 2. Lokal veriler ---
for path in local_data_paths:
    if os.path.exists(path):
        try:
            data = load_json(path)
            if isinstance(data, list):
                converted = convert_to_chat_format(
                    data,
                    instruction_key="instruction",
                    output_key="output",
                    input_key="input",
                )
                if not converted:
                    # Farklı formatı dene
                    converted = convert_to_chat_format(
                        data,
                        instruction_key="question",
                        output_key="answer",
                        input_key=None,
                    )
                sft_data.extend(converted)
                print(f"  ✅ {path}: {len(converted)} sohbet")
        except Exception as e:
            print(f"  ⚠️ {path}: {e}")

# --- 3. Dahili SFT verileri (minimum garanti) ---
if len(sft_data) < 50:
    print("\n📝 Minimum SFT verisi sağlanıyor...")
    builtin_sft = [
        {"messages": [
            {"role": "system", "content": "Sen yardımcı bir yapay zeka asistanısın."},
            {"role": "user", "content": "Python'da bir listeyi nasıl sıralarım?"},
            {"role": "assistant", "content": "Python'da listeyi sıralamak için iki yöntem vardır:\n\n1. **`sort()` metodu** — Listeyi yerinde sıralar:\n```python\nmy_list = [3, 1, 4, 1, 5]\nmy_list.sort()\nprint(my_list)  # [1, 1, 3, 4, 5]\n```\n\n2. **`sorted()` fonksiyonu** — Yeni bir sıralı liste döndürür:\n```python\nmy_list = [3, 1, 4, 1, 5]\nsorted_list = sorted(my_list)\n```\n\nTersten sıralama için `reverse=True` parametresini kullanabilirsiniz."}
        ]},
        {"messages": [
            {"role": "system", "content": "Sen yardımcı bir yapay zeka asistanısın."},
            {"role": "user", "content": "Transformer mimarisi nedir?"},
            {"role": "assistant", "content": "Transformer, 2017'de Google tarafından geliştirilen bir derin öğrenme mimarisidir.\n\n**Temel bileşenler:**\n- **Self-Attention**: Dizideki her tokenin diğer tokenlerle ilişkisini hesaplar\n- **Multi-Head Attention**: Birden fazla attention hesaplaması paralel çalışır\n- **Position Encoding**: Sıra bilgisini modele ekler\n- **Feed-Forward Network**: Her pozisyon için ayrı dönüşüm uygular\n\n**Avantajları:**\n- Paralel işleme yeteneği (RNN'lerin aksine)\n- Uzun menzilli bağımlılıkları yakalama\n- Ölçeklenebilirlik\n\nGPT, BERT, T5, LLaMA gibi modern modeller Transformer tabanlıdır."}
        ]},
        {"messages": [
            {"role": "user", "content": "Yapay zekanın geleceği hakkında ne düşünüyorsun?"},
            {"role": "assistant", "content": "Yapay zeka hızla gelişen bir alan ve geleceği oldukça heyecan verici:\n\n1. **Genel Yapay Zeka (AGI)**: İnsan düzeyinde çok yönlü düşünebilen AI sistemleri üzerinde çalışmalar devam ediyor.\n\n2. **Sağlık**: Hastalık teşhisi, ilaç keşfi ve kişiselleştirilmiş tedavi planlarında devrim bekleniyor.\n\n3. **Eğitim**: Kişiye özel öğrenme deneyimleri ve adaptif eğitim sistemleri yaygınlaşacak.\n\n4. **Bilimsel Keşif**: Protein yapısı tahmini, iklim modelleme ve malzeme bilimi gibi alanlarda önemli katkılar sağlanacak.\n\n5. **Etik Zorluklar**: Gizlilik, önyargı ve iş gücü dönüşümü konularında düzenlemeler gerekecek.\n\nÖnemli olan, bu teknolojiyi insanlığın yararına kullanmaktır."}
        ]},
        {"messages": [
            {"role": "user", "content": "Fibonacci sayılarını hesaplayan bir fonksiyon yaz."},
            {"role": "assistant", "content": "İşte farklı yaklaşımlarla Fibonacci fonksiyonları:\n\n```python\n# 1. Recursive (basit ama yavaş)\ndef fib_recursive(n):\n    if n <= 1:\n        return n\n    return fib_recursive(n-1) + fib_recursive(n-2)\n\n# 2. Iterative (hızlı ve verimli)\ndef fib_iterative(n):\n    if n <= 1:\n        return n\n    a, b = 0, 1\n    for _ in range(2, n+1):\n        a, b = b, a + b\n    return b\n\n# 3. Memoization ile (dynamic programming)\nfrom functools import lru_cache\n\n@lru_cache(maxsize=None)\ndef fib_memo(n):\n    if n <= 1:\n        return n\n    return fib_memo(n-1) + fib_memo(n-2)\n\n# 4. Generator (ilk n Fibonacci sayısı)\ndef fib_generator(n):\n    a, b = 0, 1\n    for _ in range(n):\n        yield a\n        a, b = b, a + b\n\n# Kullanım\nprint(fib_iterative(10))  # 55\nprint(list(fib_generator(10)))  # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\n```\n\nÜretim kodunda **iterative** yaklaşım tercih edilir çünkü O(n) zaman ve O(1) bellek kullanır."}
        ]},
        {"messages": [
            {"role": "user", "content": "Atatürk'ün hayatı hakkında bilgi ver."},
            {"role": "assistant", "content": "**Mustafa Kemal Atatürk** (1881-1938), Türkiye Cumhuriyeti'nin kurucusu ve ilk cumhurbaşkanıdır.\n\n**Hayatı:**\n- 1881'de Selanik'te doğdu\n- Askeri okullarda eğitim aldı, genç yaşta 'Kemal' lakabını aldı\n- Çanakkale Savaşı'nda (1915) üstün liderlik gösterdi\n- 19 Mayıs 1919'da Samsun'a çıkarak Kurtuluş Savaşı'nı başlattı\n\n**Devrimleri:**\n- Cumhuriyetin ilanı (29 Ekim 1923)\n- Halifeliğin kaldırılması\n- Harf Devrimi (Latin alfabesi)\n- Medeni Kanun\n- Kadınlara seçme ve seçilme hakkı\n- Ölçü ve takvim reformları\n- Eğitim reformları\n\n10 Kasım 1938'de İstanbul'da vefat etmiştir. 'Yurtta sulh, cihanda sulh' ilkesi onun en bilinen sözlerinden biridir."}
        ]},
    ]
    sft_data.extend(builtin_sft)

print(f"\n{'=' * 60}")
print(f"📊 SFT VERİ ÖZETİ")
print(f"{'=' * 60}")
print(f"  Toplam sohbet sayısı: {len(sft_data)}")

In [ ]:
# ═══════════════════════════════════════════════════════
# 3c. CoT (Chain-of-Thought) Verisi
# ═══════════════════════════════════════════════════════
print("🧠 CoT (Chain-of-Thought) verisi hazırlanıyor...")

cot_data = []

# --- 1. Yeni format CoT verisi (instruction/thinking/response) ---
cot_paths = [
    "data/cot/cot_training_data.json",
    "data/cot/turkish_cot_data.json",
]

for path in cot_paths:
    if os.path.exists(path):
        try:
            data = load_json(path)
            if isinstance(data, list):
                cot_data.extend(data)
                print(f"  ✅ {path}: {len(data)} örnek")
        except Exception as e:
            print(f"  ⚠️ {path}: {e}")

# --- 2. Eski format CoT verisi (question/reasoning/answer) dönüştür ---
old_cot_path = "data/examples/cot_dataset.json"
if os.path.exists(old_cot_path):
    try:
        old_data = load_json(old_cot_path)
        if isinstance(old_data, list):
            converted_cot = convert_cot_format(
                old_data,
                question_key="question",
                reasoning_key="reasoning",
                answer_key="answer",
            )
            cot_data.extend(converted_cot)
            print(f"  ✅ {old_cot_path}: {len(converted_cot)} örnek (dönüştürüldü)")
    except Exception as e:
        print(f"  ⚠️ {old_cot_path}: {e}")

# --- 3. Minimum CoT verisi garanti ---
if len(cot_data) < 10:
    print("\n📝 Minimum CoT verisi sağlanıyor...")
    builtin_cot = [
        {
            "instruction": "15 ile 27'nin toplamını hesapla.",
            "thinking": "İki sayıyı toplamamız gerekiyor.\n15 + 27\nÖnce birleri toplayalım: 5 + 7 = 12, 2 yaz 1 elde\nOnları toplayalım: 1 + 2 + 1(elde) = 4\nSonuç: 42",
            "response": "15 + 27 = 42"
        },
        {
            "instruction": "Aşağıdaki Python kodundaki hatayı bul:\ndef ortalama(liste):\n    toplam = sum(liste)\n    return toplam / len(liste)",
            "thinking": "Kodu inceliyorum:\n1. `sum(liste)` — toplam hesaplanıyor, tamam\n2. `len(liste)` — eleman sayısını alıyor\n3. Peki ya liste boşsa? `len(liste)` sıfır olur ve ZeroDivisionError hatası verir!\nBoş liste kontrolü yapılmalı.",
            "response": "Kodda boş liste kontrolü eksik. Liste boş olduğunda `ZeroDivisionError` hatası verir.\n\nDüzeltilmiş hali:\n```python\ndef ortalama(liste):\n    if not liste:\n        return 0\n    toplam = sum(liste)\n    return toplam / len(liste)\n```"
        },
        {
            "instruction": "Neden gökyüzü mavidir?",
            "thinking": "Bu atmosferik fizikle ilgili bir soru.\n\n1. Güneş ışığı beyaz ışıktan oluşur — tüm renklerin karışımı\n2. Atmosfere girdiğinde gaz molekülleriyle etkileşir\n3. Rayleigh saçılması: Kısa dalga boylu ışık (mavi) uzun dalga boylu ışıktan (kırmızı) daha fazla saçılır\n4. Mavi ışığın dalga boyu ~450nm, kırmızının ~700nm\n5. Saçılma oranı dalga boyunun 4. kuvvetiyle ters orantılı\n6. Bu nedenle mavi ışık her yönde daha fazla saçılır\n7. Gün batımında ise ışık daha uzun yol kat eder, mavi tamamen saçılır, kırmızı/turuncu kalır",
            "response": "Gökyüzü mavi görünür çünkü **Rayleigh saçılması** olur. Güneş ışığı atmosferdeki gaz molekülleriyle etkileştiğinde, kısa dalga boylu mavi ışık, uzun dalga boylu kırmızı ışıktan çok daha fazla saçılır. Bu saçılma her yöne olduğu için gökyüzüne baktığımızda mavi renk görürüz. Gün batımında ise ışık daha uzun yol kat ettiğinden mavi tamamen saçılır ve kırmızı-turuncu tonlar baskın olur."
        },
    ]
    cot_data.extend(builtin_cot)

print(f"\n{'=' * 60}")
print(f"📊 COT VERİ ÖZETİ")
print(f"{'=' * 60}")
print(f"  Toplam CoT örneği: {len(cot_data)}")
print(f"  Toplam eğitim verisi: {len(pretrain_texts)} metin + {len(sft_data)} sohbet + {len(cot_data)} CoT")

## 4️⃣ Tokenizer Eğitimi

SentencePiece BPE tokenizer'ı eğitim verisi üzerinde eğitiyoruz.
Tokenizer, Türkçe karakter setini ve özel tokenları (CoT dahil) destekler.

In [ ]:
# ═══════════════════════════════════════════════════════
# Tokenizer Eğitimi
# ═══════════════════════════════════════════════════════
print("🔤 Tokenizer eğitiliyor...")

TOKENIZER_DIR = "trained_tokenizer"
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer = ModernTokenizer(vocab_size=model_config.vocab_size)

# Tüm eğitim metinlerini birleştir
all_texts_for_tokenizer = pretrain_texts.copy()

# SFT verisinden de metin çıkar
for item in sft_data:
    for msg in item.get("messages", []):
        if msg.get("content"):
            all_texts_for_tokenizer.append(msg["content"])

# CoT verisinden de metin çıkar
for item in cot_data:
    for key in ["instruction", "thinking", "response"]:
        if item.get(key):
            all_texts_for_tokenizer.append(item[key])

print(f"  Tokenizer eğitim metni: {len(all_texts_for_tokenizer)} metin")

# Tokenizer'ı eğit
try:
    tokenizer.train(
        texts=all_texts_for_tokenizer,
        vocab_size=model_config.vocab_size,
        output_path=TOKENIZER_DIR,
    )
    print(f"  ✅ SentencePiece BPE tokenizer eğitildi")

    # Kaydet
    tokenizer.save(TOKENIZER_DIR)
    print(f"  💾 Tokenizer kaydedildi: {TOKENIZER_DIR}/")
except Exception as e:
    print(f"  ⚠️ SentencePiece eğitimi başarısız: {e}")
    print(f"  📌 Byte-level fallback tokenizer kullanılacak")

# Test
test_text = "Merhaba, ben sıfırdan yazılmış bir dil modeliyim!"
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens)
print(f"\n🧪 Tokenizer Testi:")
print(f"  Orijinal: {test_text}")
print(f"  Token ID: {tokens[:20]}{'...' if len(tokens) > 20 else ''}")
print(f"  Token sayısı: {len(tokens)}")
print(f"  Decoded: {decoded}")

# Özel token testi
print(f"\n🏷️ Özel Tokenlar:")
print(f"  <bos> = {tokenizer.bos_token_id}")
print(f"  <eos> = {tokenizer.eos_token_id}")
print(f"  <pad> = {tokenizer.pad_token_id}")
print(f"  <think> = {tokenizer.special_tokens.get('<think>', 'N/A')}")
print(f"  </think> = {tokenizer.special_tokens.get('</think>', 'N/A')}")

## 5️⃣ Model Oluşturma

GPU'ya göre seçilen konfigürasyonla modeli oluşturuyoruz.
Model özeti ve bellek tahmini gösterilir.

In [ ]:
# ═══════════════════════════════════════════════════════
# Model Oluşturma
# ═══════════════════════════════════════════════════════
print("🏗️ Model oluşturuluyor...")

# Modeli oluştur
model = ModernLLMForCausalLM(model_config)

# GPU'ya taşı
model = model.to(device)

# Model özeti
print_model_summary(model, model_config)

# Bellek tahmini
param_info = count_parameters(model)
mem_estimate = estimate_memory(
    model_config,
    batch_size=train_config.batch_size,
    seq_length=train_config.max_seq_length,
)

print(f"\n{'=' * 60}")
print(f"📊 MODEL İSTATİSTİKLERİ")
print(f"{'=' * 60}")
print(f"  Toplam parametre: {format_params(param_info['total'])}")
print(f"  Eğitilebilir parametre: {format_params(param_info['trainable'])}")
print(f"  Tahmini bellek (model): {mem_estimate.get('model_memory_gb', 'N/A'):.2f} GB")
print(f"  Tahmini bellek (eğitim): {mem_estimate.get('total_training_memory_gb', 'N/A'):.2f} GB")
print(f"  GPU VRAM: {GPU_MEMORY_GB:.1f} GB")

if GPU_MEMORY_GB > 0:
    total_needed = mem_estimate.get('total_training_memory_gb', 0)
    if total_needed < GPU_MEMORY_GB * 0.9:
        print(f"  ✅ VRAM yeterli!")
    elif total_needed < GPU_MEMORY_GB:
        print(f"  ⚠️ VRAM sınırda, OOM riski var")
    else:
        print(f"  ❌ VRAM yetersiz! Daha küçük model veya batch size deneyin")

# Gradient checkpointing (bellek tasarrufu)
if GPU_MEMORY_GB < 16:
    model.model.gradient_checkpointing = True
    print(f"  🔄 Gradient checkpointing aktif (bellek tasarrufu)")

## 6️⃣ Aşama 1: Pre-training (Causal LM)

İlk aşamada modeli ham metin üzerinde next-token prediction ile eğitiyoruz.
Bu aşama modelin dil bilgisini ve genel bilgisini öğrenmesini sağlar.

In [ ]:
# ═══════════════════════════════════════════════════════
# Aşama 1: Pre-training
# ═══════════════════════════════════════════════════════
print("🚀 AŞAMA 1: Pre-training başlıyor...")
print("=" * 60)

# Dataset oluştur
pretrain_dataset = TextDataset(
    texts=pretrain_texts,
    tokenizer=tokenizer,
    max_length=train_config.max_seq_length,
)
print(f"  📚 Pre-training dataset: {len(pretrain_dataset)} blok")

if len(pretrain_dataset) == 0:
    print("  ⚠️ Yeterli pre-training verisi yok, bu aşama atlanıyor!")
else:
    # Eval split
    eval_size = max(1, int(len(pretrain_dataset) * 0.02))
    train_size = len(pretrain_dataset) - eval_size
    pretrain_train, pretrain_eval = torch.utils.data.random_split(
        pretrain_dataset, [train_size, eval_size]
    )

    # Eğitim konfigürasyonu
    pretrain_config = TrainingConfig(
        num_epochs=2,
        batch_size=train_config.batch_size,
        gradient_accumulation_steps=train_config.gradient_accumulation_steps,
        learning_rate=3e-4,
        warmup_ratio=0.1,
        weight_decay=0.1,
        max_grad_norm=1.0,
        mixed_precision=train_config.mixed_precision,
        max_seq_length=train_config.max_seq_length,
        logging_steps=10,
        eval_steps=50,
        save_steps=100,
        save_total_limit=2,
        output_dir="checkpoints/pretrain",
        use_wandb=False,
        stage="pretrain",
    )

    # Collate function
    collate_fn = create_data_collator(tokenizer, max_length=train_config.max_seq_length)

    # Trainer oluştur
    pretrain_trainer = Trainer(
        model=model,
        train_dataset=pretrain_train,
        eval_dataset=pretrain_eval,
        training_config=pretrain_config,
        tokenizer=tokenizer,
        collate_fn=collate_fn,
    )

    # Eğit!
    pretrain_trainer.train()

    print(f"\n✅ Pre-training tamamlandı!")

# Bellek temizle
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7️⃣ Aşama 2: SFT (Supervised Fine-Tuning)

İkinci aşamada modeli chat formatında instruction-following verisi ile fine-tune ediyoruz.
Bu aşama modelin talimatları anlaması ve uygun cevaplar vermesini sağlar.

In [ ]:
# ═══════════════════════════════════════════════════════
# Aşama 2: SFT (Supervised Fine-Tuning)
# ═══════════════════════════════════════════════════════
print("🚀 AŞAMA 2: SFT (Instruction Tuning) başlıyor...")
print("=" * 60)

# Dataset oluştur
sft_dataset = ChatDataset(
    conversations=sft_data,
    tokenizer=tokenizer,
    max_length=train_config.max_seq_length,
)
print(f"  📚 SFT dataset: {len(sft_dataset)} örnek")

if len(sft_dataset) == 0:
    print("  ⚠️ Yeterli SFT verisi yok, bu aşama atlanıyor!")
else:
    # Eval split
    eval_size = max(1, int(len(sft_dataset) * 0.05))
    train_size = len(sft_dataset) - eval_size
    sft_train, sft_eval = torch.utils.data.random_split(
        sft_dataset, [train_size, eval_size]
    )

    # SFT konfigürasyonu (daha düşük LR)
    sft_config = TrainingConfig(
        num_epochs=3,
        batch_size=train_config.batch_size,
        gradient_accumulation_steps=train_config.gradient_accumulation_steps,
        learning_rate=2e-5,  # SFT için daha düşük LR
        warmup_ratio=0.05,
        weight_decay=0.01,
        max_grad_norm=1.0,
        mixed_precision=train_config.mixed_precision,
        max_seq_length=train_config.max_seq_length,
        logging_steps=10,
        eval_steps=50,
        save_steps=100,
        save_total_limit=2,
        output_dir="checkpoints/sft",
        use_wandb=False,
        stage="sft",
    )

    # Collate function
    collate_fn = create_data_collator(tokenizer, max_length=train_config.max_seq_length)

    # Trainer oluştur
    sft_trainer = Trainer(
        model=model,
        train_dataset=sft_train,
        eval_dataset=sft_eval,
        training_config=sft_config,
        tokenizer=tokenizer,
        collate_fn=collate_fn,
    )

    # Eğit!
    sft_trainer.train()

    print(f"\n✅ SFT tamamlandı!")

# Bellek temizle
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8️⃣ Aşama 3: CoT Fine-tuning (Chain-of-Thought)

Üçüncü aşamada modeli Chain-of-Thought (akıl yürütme) verisi ile eğitiyoruz.
Bu aşama modelin `<think>...</think>` blokları içinde adım adım düşünmesini sağlar.
Modern reasoning modelleri (Claude, DeepSeek-R1 vb.) gibi çalışır.

In [ ]:
# ═══════════════════════════════════════════════════════
# Aşama 3: CoT Fine-tuning
# ═══════════════════════════════════════════════════════
print("🚀 AŞAMA 3: CoT (Chain-of-Thought) Fine-tuning başlıyor...")
print("=" * 60)

# Dataset oluştur
cot_dataset = CoTDataset(
    data=cot_data,
    tokenizer=tokenizer,
    max_length=train_config.max_seq_length,
)
print(f"  📚 CoT dataset: {len(cot_dataset)} örnek")

if len(cot_dataset) == 0:
    print("  ⚠️ Yeterli CoT verisi yok, bu aşama atlanıyor!")
else:
    # Eval split
    eval_size = max(1, int(len(cot_dataset) * 0.1))
    train_size = len(cot_dataset) - eval_size
    cot_train, cot_eval = torch.utils.data.random_split(
        cot_dataset, [train_size, eval_size]
    )

    # CoT konfigürasyonu (düşük LR, dikkatli fine-tuning)
    cot_config = TrainingConfig(
        num_epochs=5,  # CoT verisi genelde az, daha fazla epoch
        batch_size=max(1, train_config.batch_size // 2),  # Daha küçük batch
        gradient_accumulation_steps=train_config.gradient_accumulation_steps * 2,
        learning_rate=1e-5,  # CoT için en düşük LR
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=0.5,  # Daha agresif gradient clipping
        mixed_precision=train_config.mixed_precision,
        max_seq_length=train_config.max_seq_length,
        logging_steps=5,
        eval_steps=25,
        save_steps=50,
        save_total_limit=2,
        output_dir="checkpoints/cot",
        use_wandb=False,
        stage="cot",
    )

    # Collate function
    collate_fn = create_data_collator(tokenizer, max_length=train_config.max_seq_length)

    # Trainer oluştur
    cot_trainer = Trainer(
        model=model,
        train_dataset=cot_train,
        eval_dataset=cot_eval,
        training_config=cot_config,
        tokenizer=tokenizer,
        collate_fn=collate_fn,
    )

    # Eğit!
    cot_trainer.train()

    print(f"\n✅ CoT fine-tuning tamamlandı!")

# Bellek temizle
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 9️⃣ Değerlendirme ve Test

Eğitilmiş modeli test ediyoruz:
- Normal metin üretme
- Chat/sohbet modu
- CoT (düşünme) modu

In [ ]:
# ═══════════════════════════════════════════════════════
# Model Değerlendirmesi
# ═══════════════════════════════════════════════════════
print("🧪 MODEL DEĞERLENDİRMESİ")
print("=" * 60)

# TextGenerator oluştur
generator = TextGenerator(
    model=model,
    tokenizer=tokenizer,
    device=device,
)

# ─── Test 1: Basit metin tamamlama ───
print("\n📝 TEST 1: Metin Tamamlama")
print("-" * 40)

test_prompts = [
    "Yapay zeka",
    "Türkiye'nin başkenti",
    "Python programlama dilinde",
    "Transformer mimarisi",
]

for prompt in test_prompts:
    try:
        output = generator.generate(
            prompt=prompt,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
        )
        print(f"\n💬 Prompt: {prompt}")
        print(f"🤖 Çıktı: {output[:200]}{'...' if len(output) > 200 else ''}")
    except Exception as e:
        print(f"\n💬 Prompt: {prompt}")
        print(f"❌ Hata: {e}")

# ─── Test 2: Chat modu ───
print(f"\n\n{'=' * 60}")
print("💬 TEST 2: Chat Modu")
print("-" * 40)

chat_tests = [
    "Merhaba, nasılsın?",
    "Python'da list comprehension nasıl kullanılır?",
    "Dünyanın en yüksek dağı hangisidir?",
]

for user_msg in chat_tests:
    try:
        messages = [
            {"role": "system", "content": "Sen yardımcı bir yapay zeka asistanısın. Türkçe yanıt ver."},
            {"role": "user", "content": user_msg},
        ]
        
        # Chat template uygula
        chat_prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        
        output = generator.generate(
            prompt=chat_prompt,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
        )
        
        # Assistant cevabını çıkar
        if "<|assistant|>" in output:
            response = output.split("<|assistant|>")[-1].strip()
        else:
            response = output[len(chat_prompt):].strip()
        
        # Özel tokenleri temizle
        for token in ["<|im_end|>", "<|im_start|>", "<eos>", "</s>"]:
            response = response.replace(token, "").strip()
        
        print(f"\n👤 Kullanıcı: {user_msg}")
        print(f"🤖 Asistan: {response[:300]}{'...' if len(response) > 300 else ''}")
    except Exception as e:
        print(f"\n👤 Kullanıcı: {user_msg}")
        print(f"❌ Hata: {e}")

# ─── Test 3: CoT (Düşünme) modu ───
print(f"\n\n{'=' * 60}")
print("🧠 TEST 3: CoT (Chain-of-Thought) Modu")
print("-" * 40)

cot_tests = [
    "15 * 23 kaçtır? Adım adım hesapla.",
    "Bu Python kodundaki hatayı bul: def f(x): return x / 0",
]

for question in cot_tests:
    try:
        result = generator.generate_with_thinking(
            prompt=question,
            max_new_tokens=300,
            temperature=0.3,  # Düşünme için düşük temperature
        )
        
        print(f"\n❓ Soru: {question}")
        if isinstance(result, dict):
            if result.get("thinking"):
                print(f"💭 Düşünme: {result['thinking'][:200]}...")
            print(f"✅ Cevap: {result.get('response', result.get('text', ''))[:200]}")
        else:
            print(f"🤖 Çıktı: {str(result)[:300]}")
    except Exception as e:
        print(f"\n❓ Soru: {question}")
        print(f"❌ Hata: {e}")

print(f"\n{'=' * 60}")
print("✅ Değerlendirme tamamlandı!")

## 🔟 Modeli Kaydetme

Eğitilmiş modeli ve tokenizer'ı kaydedin.
Google Drive'a veya HuggingFace Hub'a yükleyebilirsiniz.

In [ ]:
# ═══════════════════════════════════════════════════════
# Model Kaydetme
# ═══════════════════════════════════════════════════════
print("💾 MODEL KAYDEDİLİYOR...")
print("=" * 60)

SAVE_DIR = "trained_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Model kaydet
model.save_pretrained(SAVE_DIR)
print(f"  ✅ Model kaydedildi: {SAVE_DIR}/")

# Tokenizer kaydet
tokenizer.save(os.path.join(SAVE_DIR, "tokenizer"))
print(f"  ✅ Tokenizer kaydedildi: {SAVE_DIR}/tokenizer/")

# Model config kaydet
save_json(model_config.to_dict(), os.path.join(SAVE_DIR, "model_config.json"))
print(f"  ✅ Model config kaydedildi")

# Eğitim bilgilerini kaydet
training_info = {
    "model_name": model_config.model_name,
    "total_params": format_params(count_parameters(model)['total']),
    "gpu_type": GPU_TYPE,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "stages_completed": ["pretrain", "sft", "cot"],
    "pretrain_texts": len(pretrain_texts),
    "sft_conversations": len(sft_data),
    "cot_examples": len(cot_data),
    "architecture": {
        "hidden_size": model_config.hidden_size,
        "num_layers": model_config.num_layers,
        "num_heads": model_config.num_attention_heads,
        "num_kv_heads": model_config.num_kv_heads,
        "intermediate_size": model_config.intermediate_size,
        "max_seq_length": model_config.max_position_embeddings,
        "features": [
            "RMSNorm", "RoPE", "GQA", "SwiGLU",
            "Flash Attention", "KV-Cache", "CoT"
        ],
    },
}
save_json(training_info, os.path.join(SAVE_DIR, "training_info.json"))
print(f"  ✅ Eğitim bilgileri kaydedildi")

# Dosya boyutları
total_size = 0
for root, dirs, files in os.walk(SAVE_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        total_size += size

print(f"\n📦 Toplam model boyutu: {total_size / 1024**2:.1f} MB")
print(f"📂 Kayıt dizini: {os.path.abspath(SAVE_DIR)}")

In [ ]:
# ═══════════════════════════════════════════════════════
# Google Drive'a Yükleme (İsteğe Bağlı)
# ═══════════════════════════════════════════════════════
if IN_COLAB:
    save_to_drive = True  # False yaparak atlayabilirsiniz
    
    if save_to_drive:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            
            drive_dir = '/content/drive/MyDrive/ModernLLM'
            os.makedirs(drive_dir, exist_ok=True)
            
            # Modeli Drive'a kopyala
            import shutil
            shutil.copytree(SAVE_DIR, os.path.join(drive_dir, 'trained_model'), dirs_exist_ok=True)
            
            # Tokenizer'ı da kopyala
            if os.path.exists(TOKENIZER_DIR):
                shutil.copytree(TOKENIZER_DIR, os.path.join(drive_dir, 'tokenizer'), dirs_exist_ok=True)
            
            print(f"✅ Model Google Drive'a kaydedildi: {drive_dir}")
        except Exception as e:
            print(f"⚠️ Drive kaydetme hatası: {e}")
    else:
        print("ℹ️ Drive'a kaydetme atlandı")
else:
    print("ℹ️ Colab ortamında değil, Drive kaydetme atlanıyor")

## 📚 Modeli Yükleme ve Kullanma

Eğitilmiş modeli daha sonra nasıl yükleyeceğiniz:

In [ ]:
# ═══════════════════════════════════════════════════════
# Eğitilmiş Modeli Yükleme ve Kullanma
# ═══════════════════════════════════════════════════════
print("📦 Eğitilmiş model yükleniyor...")

# Model yükle
loaded_model = ModernLLMForCausalLM.from_pretrained(SAVE_DIR, device=str(device))
print(f"  ✅ Model yüklendi: {format_params(count_parameters(loaded_model)['total'])} parametre")

# Tokenizer yükle
loaded_tokenizer = ModernTokenizer()
tokenizer_path = os.path.join(SAVE_DIR, "tokenizer")
if os.path.exists(tokenizer_path):
    loaded_tokenizer.load(tokenizer_path)
    print(f"  ✅ Tokenizer yüklendi")
else:
    loaded_tokenizer = tokenizer
    print(f"  ℹ️ Original tokenizer kullanılıyor")

# Generator oluştur
gen = TextGenerator(
    model=loaded_model,
    tokenizer=loaded_tokenizer,
    device=device,
)

# Test
print("\n🧪 Yüklenen model testi:")
response = gen.generate(
    prompt="Yapay zeka nedir?",
    max_new_tokens=150,
    temperature=0.7,
)
print(f"  🤖 {response[:300]}")

print(f"\n{'=' * 60}")
print("🎉 TÜM AŞAMALAR TAMAMLANDI!")
print(f"{'=' * 60}")
print(f"""
📋 Özet:
  • Model: {model_config.model_name}
  • Parametre: {format_params(count_parameters(model)['total'])}
  • GPU: {GPU_TYPE} ({GPU_MEMORY_GB:.1f} GB)
  • Eğitim aşamaları: Pre-training → SFT → CoT
  • Veri: {len(pretrain_texts)} metin + {len(sft_data)} sohbet + {len(cot_data)} CoT

🚀 Modeli kullanmak için:
  from modern_llm.model.transformer import ModernLLMForCausalLM
  from modern_llm.tokenizer import ModernTokenizer
  from modern_llm.inference.generator import TextGenerator

  model = ModernLLMForCausalLM.from_pretrained("{SAVE_DIR}")
  tokenizer = ModernTokenizer(model_path="{SAVE_DIR}/tokenizer")
  generator = TextGenerator(model, tokenizer)
  output = generator.generate("Merhaba!")
""")